In [0]:
%sql
create table if not exists books_silver
(book_id string, title string, author string, price LONGINT, current boolean, effective_date timestamp, end_date timestamp)

In [0]:
def type2_upsert(microBatchDF, batch):
    microBatchDF.createOrReplaceTempView("updates")

    sql_query = """
    MERGE INTO books_silver 
    using(
        select book_id as merge_key, updates.*
        from updates
        union all
        select null as merge_key, updates.*
        from updates
        join books_silver on updates.book_id = books_silver.book_id
        where books_silver.current = true and updates.price <> books_silver.price 
        ) staged_updates
    on books_silver.book_id = merge_key
    when matched and books_silver.current = true and books_silver.price <> staged_updates.price then
    update set current = false and end_date = staged_updates.updated
    when not matched then insert 
    (book_id, title, author, price, current, effective_date, end_date)
    values(staged_updates.book_id, staged_updates.title, staged_updates.author, staged_updates.price, true, staged_updates.updated, NULL)
    """

    microBatchDF.sparkSession.sql(sql_query)

In [0]:
def process_bronze():
    books_schema = "book_id string, title string, author string, price LONGINT, updated timestamp"

    query = (spark.readStream
              .table("workspace.bookstore_eng_pro.bronze")
              .filter("topic = 'books'")
              .select(F.from_json(F.unbase64(F.col("value")).cast("string"),books_schema).alias("v"))
              .select("v.*")
              .writeStream
              .foreachBatch("type2_upsert")
              .option("checkpointLocation", "/Volumes/workspace/bookstore_eng_pro/checkpoints/books_silver")
              .trigger(availableNow=True)
              .start())
    query.awaitTermination()

In [0]:
%sql
select cast(unbase64(value) as string)
from workspace.bookstore_eng_pro.bronze
where topic = 'books'